<a href="https://colab.research.google.com/github/IDGS-904-23001532/ECOSYSTEM/blob/main/smart_ecology_ecosystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Base de datos 1.0

In [ ]:
import sqlite3

# 1. Conectar a la base de datos local (actuará como búfer en la escuela)
conn = sqlite3.connect("data_warehouse_solar.db")
cursor = conn.cursor()

# Habilitar soporte de llaves foráneas en SQLite para garantizar la integridad referencial
cursor.execute("PRAGMA foreign_keys = ON;")


¡Conexión exitosa a SQLite! Creando tablas adaptadas al diagrama final...


In [ ]:
# Tabla de tiempo
cursor.execute("""
CREATE TABLE IF NOT EXISTS Dim_Tiempo (
    ID_Tiempo INTEGER PRIMARY KEY,
    Fecha_Completa TEXT NOT NULL,       -- Almacena formato ISO (Ej. '2026-05-23')
    Periodo_Dia TEXT                    -- Ej. 'Mañana', 'Tarde', 'Noche'
);
""")

# Tabla de disposivos Dispositivo
cursor.execute("""
CREATE TABLE IF NOT EXISTS Dim_Dispositivo (
    ID_Dispositivo INTEGER PRIMARY KEY,
    Identificador_Nodo TEXT NOT NULL,   -- Código único del hardware/nodo
    Tipo_Dispositivo TEXT,              -- Ej. 'Panel Solar', 'Inversor', 'Medidor'
    Categoria TEXT,                     -- 'Generación' o 'Consumo'
    Estado_Operativo TEXT               -- Ej. 'Activo', 'Mantenimiento'
);
""")

# Tablade de Modo Sistema
cursor.execute("""
CREATE TABLE IF NOT EXISTS Dim_Modo_Sistema (
    ID_Modo INTEGER PRIMARY KEY,
    Nombre_Modo TEXT NOT NULL,          -- Ej. 'Ahorro Energía (Eco)', 'Normal'
    Prioridad INTEGER                   -- Nivel numérico de prioridad de atención
);
""")

In [ ]:
# =====================================================================
# 4. CREACIÓN DE LA TABLA DE HECHOS (Fact_Lecturas_IoT con Store-and-Forward)
# =====================================================================
cursor.execute("""
CREATE TABLE IF NOT EXISTS Fact_Lecturas_IoT (
    ID_Lectura INTEGER PRIMARY KEY AUTOINCREMENT,
    ID_Tiempo INTEGER NOT NULL,
    ID_Dispositivo INTEGER NOT NULL,
    ID_Modo INTEGER NOT NULL,

    -- Métricas del Negocio (Variables de medición continua)
    Valor_Voltaje REAL,
    Valor_Corriente REAL,
    Valor_Temperatura REAL,
    Porcentaje_Bateria REAL,
    Energia_Generada_Wh REAL,           -- Métricas diferenciadas (Panel vs Casa)
    Energia_Consumida_Wh REAL,
    Angulo_Panel_H_V TEXT,              -- Orientación física del panel
    Alerta_Generada INTEGER,            -- 0 = Sin alerta, 1 = Alerta activa
    Tipo_Alerta TEXT,                   -- Ej. 'Sobrevoltaje', 'Batería Crítica'

    -- Control de Estrategia: Mitigación de Internet Inestable de la Escuela
    Estado_Sincronizacion INTEGER DEFAULT 0, -- 0 = Pendiente (Búfer Local), 1 = Subido a Nube
    Fecha_Captura_Local TEXT DEFAULT CURRENT_TIMESTAMP,

    FOREIGN KEY (ID_Tiempo) REFERENCES Dim_Tiempo(ID_Tiempo),
    FOREIGN KEY (ID_Dispositivo) REFERENCES Dim_Dispositivo(ID_Dispositivo),
    FOREIGN KEY (ID_Modo) REFERENCES Dim_Modo_Sistema(ID_Modo)
);
""")

In [ ]:
# Inicializar Parámetros en Español (Sustento de Ingeniería de Datos)
parametros = [
    ("Frecuencia_Muestreo_Seg", 60.0, "Segundos", "Intervalo base de toma de lecturas en los nodos"),
    ("Umbral_Zona_Muerta", 5.0, "Porcentaje", "Porcentaje mínimo de cambio para forzar transmisión"),
    ("Ventana_Agrupacion_Borde", 5.0, "Minutos", "Tiempo de compactación local en memoria antes del volcado"),
    ("cfg_alerta_umbral_bateria", 20.0, "Porcentaje", "Límite crítico para activar el cómputo ligero local")
]

In [ ]:
# Poblado de Dim_Modo_Sistema
modos = [
    (1, "Ahorro de Energía (Eco)", 1),
    (2, "Carga Rápida (Grid)", 2),
    (3, "Respaldo de Emergencia", 3)
]
cursor.executemany("INSERT OR IGNORE INTO Dim_Modo_Sistema VALUES (?, ?, ?);", modos)

# Poblado de Dim_Dispositivo (Panel Solar [Generación] y Medidor de la Casa [Consumo])
dispositivos = [
    (101, "INV-GROWATT-01", "Inversor Híbrido", "Generación", "Activo"),
    (102, "MED-SONOFF-02", "Sensor Consumo Residencia", "Consumo", "Activo")
]
cursor.executemany("INSERT OR IGNORE INTO Dim_Dispositivo VALUES (?, ?, ?, ?, ?);", dispositivos)

# Poblado de Dim_Tiempo
tiempos = [
    (2026052301, "2026-05-23", "Mediodía"),
    (2026052302, "2026-05-23", "Tarde")
]
cursor.executemany("INSERT OR IGNORE INTO Dim_Tiempo VALUES (?, ?, ?);", tiempos)

# Inserción en la Tabla de Hechos (Métrica cruzada Panel Solar vs Consumo Casa)
# Registro 1: El panel solar genera energía limpia al mediodía (Estado_Sincronizacion = 0)
cursor.execute("""
INSERT INTO Fact_Lecturas_IoT
(ID_Tiempo, ID_Dispositivo, ID_Modo, Valor_Voltaje, Valor_Corriente, Valor_Temperatura, Porcentaje_Bateria, Energia_Generada_Wh, Energia_Consumida_Wh, Angulo_Panel_H_V, Alerta_Generada, Tipo_Alerta)
VALUES (2026052301, 101, 1, 120.5, 3.73, 35.2, 85.0, 450.0, 0.0, 'H:15°/V:45°', 0, 'Ninguna');
""")

# Registro 2: El medidor registra el consumo de la casa por la tarde (Estado_Sincronizacion = 0)
cursor.execute("""
INSERT INTO Fact_Lecturas_IoT
(ID_Tiempo, ID_Dispositivo, ID_Modo, Valor_Voltaje, Valor_Corriente, Valor_Temperatura, Porcentaje_Bateria, Energia_Generada_Wh, Energia_Consumida_Wh, Angulo_Panel_H_V, Alerta_Generada, Tipo_Alerta)
VALUES (2026052302, 102, 1, 118.2, 1.56, 24.1, 84.5, 0.0, 185.0, 'No Aplica', 0, 'Ninguna');
""")

conn.commit()
print("¡Datos de prueba acoplados correctamente!")

¡Datos de prueba acoplados correctamente!


In [ ]:
# =====================================================================
# 7. CONTROL DE AUDITORÍA: VERIFICACIÓN DEL BÚFER LOCAL ANTES DEL ENVÍO
# =====================================================================
print("\n--- SIMULACIÓN DE CONSULTA LOCAL (DATOS EN ESPERA DE 'RED BUENA' PARA TRANSMISIÓN MASIVA) ---")
query = """
SELECT
    T.Fecha_Completa || ' (' || T.Periodo_Dia || ')' AS Bloque_Temporal,
    D.Identificador_Nodo AS Codigo_Sensor,
    D.Categoria AS Tipo_Flujo,
    F.Energia_Generada_Wh AS Gen_Wh,
    F.Energia_Consumida_Wh AS Cons_Wh,
    CASE F.Estado_Sincronizacion
        WHEN 0 THEN 'Pendiente en Búfer Local (Escuela)'
        ELSE 'Sincronizado a la Nube Exitosamente'
    END AS Estatus_Red
FROM Fact_Lecturas_IoT F
JOIN Dim_Tiempo T ON F.ID_Tiempo = T.ID_Tiempo
JOIN Dim_Dispositivo D ON F.ID_Dispositivo = D.ID_Dispositivo
WHERE F.Estado_Sincronizacion = 0;
"""

cursor.execute(query)
resultados = cursor.fetchall()

for fila in resultados:
    print(f"Tiempo: {fila[0]} | Nodo: {fila[1]} | Flujo: {fila[2]} | Gen: {fila[3]}Wh | Consumo: {fila[4]}Wh | Estado: {fila[5]}")

# Cierre seguro de la base de datos temporal
conn.close()


--- SIMULACIÓN DE CONSULTA LOCAL (DATOS EN ESPERA DE 'RED BUENA' PARA TRANSMISIÓN MASIVA) ---
Tiempo: 2026-05-23 (Mediodía) | Nodo: INV-GROWATT-01 | Flujo: Generación | Gen: 450.0Wh | Consumo: 0.0Wh | Estado: Pendiente en Búfer Local (Escuela)
Tiempo: 2026-05-23 (Tarde) | Nodo: MED-SONOFF-02 | Flujo: Consumo | Gen: 0.0Wh | Consumo: 185.0Wh | Estado: Pendiente en Búfer Local (Escuela)





# Tipos y fuentes de datos

In [ ]:
import sqlite3

# Para evitar conflictos con el esquema anterior, borramos la BD vieja si existe en Colab
# if os.path.exists("data_warehouse_solar_esquema_foto.db"):
#    os.remove("data_warehouse_solar_esquema_foto.db")

# 1. Conectar a la base de datos
conn = sqlite3.connect("data_warehouse_solar_esquema_foto.db")
cursor = conn.cursor()

# Habilitar el soporte de llaves foráneas en SQLite
cursor.execute("PRAGMA foreign_keys = ON;")

print("¡Conexión exitosa a SQLite! Creando tablas...")

# =====================================================================
# 2. CREACIÓN DE TABLAS DE DIMENSIONES
# =====================================================================

# Dimensión Tiempo
cursor.execute("""
CREATE TABLE IF NOT EXISTS Dim_Tiempo (
    ID_Tiempo INTEGER PRIMARY KEY AUTOINCREMENT,
    Fecha_Completa DATE NOT NULL,
    Anio INTEGER NOT NULL,
    Mes INTEGER NOT NULL,
    Dia INTEGER NOT NULL,
    Hora INTEGER NOT NULL,
    Minuto INTEGER NOT NULL
);
""")

# Dimensión Ubicacion
cursor.execute("""
CREATE TABLE IF NOT EXISTS Dim_Ubicacion (
    ID_Ubicacion INTEGER PRIMARY KEY AUTOINCREMENT,
    Nombre_Zona TEXT NOT NULL
);
""")

# Dimensión Dispositivo
cursor.execute("""
CREATE TABLE IF NOT EXISTS Dim_Dispositivo (
    ID_Dispositivo INTEGER PRIMARY KEY AUTOINCREMENT,
    Identificador_Nodo TEXT NOT NULL,
    Tipo_Dispositivo TEXT NOT NULL,
    Categoria TEXT NOT NULL
);
""")

# =====================================================================
# 3. CREACIÓN DE LA TABLA DE HECHOS (Fact_Lecturas_IoT)
# =====================================================================
cursor.execute("""
CREATE TABLE IF NOT EXISTS Fact_Lecturas_IoT (
    ID_Lectura INTEGER PRIMARY KEY AUTOINCREMENT,
    ID_Tiempo INTEGER NOT NULL,
    ID_Ubicacion INTEGER NOT NULL,
    ID_Dispositivo INTEGER NOT NULL,
    Valor_Voltaje REAL,
    Valor_Corriente REAL,
    Valor_Temperatura REAL,
    Porcentaje_Bateria REAL,
    FOREIGN KEY (ID_Tiempo) REFERENCES Dim_Tiempo(ID_Tiempo),
    FOREIGN KEY (ID_Ubicacion) REFERENCES Dim_Ubicacion(ID_Ubicacion),
    FOREIGN KEY (ID_Dispositivo) REFERENCES Dim_Dispositivo(ID_Dispositivo)
);
""")

print("Tablas creadas correctamente según el diagrama.")

# =====================================================================
# 4. INSERCIÓN DE DATOS DE PRUEBA (Para validar el modelo)
# =====================================================================

print("Insertando datos de prueba...")

# Insertar en Dim_Tiempo
tiempos = [
    ("2026-05-23", 2026, 5, 23, 12, 0),
    ("2026-05-23", 2026, 5, 23, 12, 1)
]
cursor.executemany("INSERT INTO Dim_Tiempo (Fecha_Completa, Anio, Mes, Dia, Hora, Minuto) VALUES (?, ?, ?, ?, ?, ?);", tiempos)

# Insertar en Dim_Ubicacion
ubicaciones = [
    ("Panel Solar - Techo",),
    ("Cochera Automatizada",)
]
cursor.executemany("INSERT INTO Dim_Ubicacion (Nombre_Zona) VALUES (?);", ubicaciones)

# Insertar en Dim_Dispositivo
dispositivos = [
    ("ESP32_NODO_01", "Sensor", "Energía"),
    ("ESP32_NODO_02", "Actuador", "Iluminación")
]
cursor.executemany("INSERT INTO Dim_Dispositivo (Identificador_Nodo, Tipo_Dispositivo, Categoria) VALUES (?, ?, ?);", dispositivos)

# Insertar en Fact_Lecturas_IoT
# Relacionamos: Tiempo 1, Ubicación 1 (Techo), Dispositivo 1 (Nodo Energía)
cursor.execute("INSERT INTO Fact_Lecturas_IoT (ID_Tiempo, ID_Ubicacion, ID_Dispositivo, Valor_Voltaje, Valor_Corriente, Valor_Temperatura, Porcentaje_Bateria) VALUES (1, 1, 1, 24.5, 5.2, 35.2, 85.0);")
# Relacionamos: Tiempo 2, Ubicación 2 (Cochera), Dispositivo 2 (Nodo Iluminación)
cursor.execute("INSERT INTO Fact_Lecturas_IoT (ID_Tiempo, ID_Ubicacion, ID_Dispositivo, Valor_Voltaje, Valor_Corriente, Valor_Temperatura, Porcentaje_Bateria) VALUES (2, 2, 2, 12.0, 1.1, 28.5, 84.5);")

# Confirmar cambios
conn.commit()
print("¡Datos de prueba insertados!")

# =====================================================================
# 5. PRUEBA DE CONSULTA (Simulación de consulta analítica/BI)
# =====================================================================
print("\n--- EJECUTANDO CONSULTA DE PRUEBA (JOIN COMPLETO) ---")
query = """
SELECT
    T.Fecha_Completa || ' ' || T.Hora || ':' || printf('%02d', T.Minuto) AS Momento,
    U.Nombre_Zona,
    D.Identificador_Nodo,
    F.Valor_Voltaje,
    F.Valor_Corriente,
    F.Valor_Temperatura
FROM Fact_Lecturas_IoT F
JOIN Dim_Tiempo T ON F.ID_Tiempo = T.ID_Tiempo
JOIN Dim_Ubicacion U ON F.ID_Ubicacion = U.ID_Ubicacion
JOIN Dim_Dispositivo D ON F.ID_Dispositivo = D.ID_Dispositivo;
"""

cursor.execute(query)
resultados = cursor.fetchall()

for fila in resultados:
    print(f"Momento: {fila[0]} | Zona: {fila[1]} | Nodo: {fila[2]} | Volts: {fila[3]}V | Amps: {fila[4]}A | Temp: {fila[5]}°C")

# Cerrar la conexión
conn.close()
print("\nBase de datos cerrada.")

¡Conexión exitosa a SQLite! Creando tablas...
Tablas creadas correctamente según el diagrama.
Insertando datos de prueba...
¡Datos de prueba insertados!

--- EJECUTANDO CONSULTA DE PRUEBA (JOIN COMPLETO) ---
Momento: 2026-05-23 12:00 | Zona: Panel Solar - Techo | Nodo: ESP32_NODO_01 | Volts: 24.5V | Amps: 5.2A | Temp: 35.2°C
Momento: 2026-05-23 12:01 | Zona: Cochera Automatizada | Nodo: ESP32_NODO_02 | Volts: 12.0V | Amps: 1.1A | Temp: 28.5°C

Base de datos cerrada.




# TÉCNICAS DE LIMPIEZA DE DATOS

In [ ]:
import json

# =====================================================================
# 1. SEMIESTRUCTURADOS / EN TIEMPO REAL (Streaming) / SENSORES IoT
# Ahora coincide con todas las columnas de Fact_Lecturas_IoT
# =====================================================================
datos_mqtt = {
  "timestamp": "2026-05-23T12:00:00Z",
  "id_dispositivo_origen": "NODO_ESP32_CENTRAL",
  "modo_operativo": "Automático Inteligente",
  "lecturas_tiempo_real": {
    "valor_voltaje": 5.1,
    "valor_corriente": 0.45,
    "valor_temperatura": 35.2,
    "porcentaje_bateria": 85.0,
    "energia_generada_wh": 450.0,
    "energia_consumida_wh": 120.0,
    "angulo_panel_h_v": "H: 90, V: 45"
  },
  "estado_alertas": {
    "alerta_generada": 0,
    "tipo_alerta": "Ninguna"
  }
}
with open('1_semiestructurados_streaming_iot.json', 'w', encoding='utf-8') as f:
    json.dump(datos_mqtt, f, indent=4, ensure_ascii=False)


# =====================================================================
# 2. ESTRUCTURADOS / SERIES TEMPORALES / BASE DE DATOS INTERNA
# Ahora incluye todas las variables de la tabla de hechos
# =====================================================================
csv_series_temporales = """fecha_hora,voltaje,corriente,temperatura,bateria,energia_gen_wh,energia_cons_wh,angulo_panel,alerta_activa,tipo_alerta
2026-05-23 12:00:00,5.1,0.45,35.2,85.0,450.0,120.0,"H: 90, V: 45",0,Ninguna
2026-05-23 12:05:00,5.1,0.46,35.5,85.5,455.0,122.5,"H: 92, V: 45",0,Ninguna
2026-05-23 12:10:00,4.8,0.10,28.5,95.5,120.0,80.0,"H: 180, V: 30",1,Baja Captación Solar"""
with open('2_estructurados_series_temporales.csv', 'w', encoding='utf-8') as f:
    f.write(csv_series_temporales)


# =====================================================================
# 3. DATOS MAESTROS
# Las columnas ahora coinciden exactamente con Dim_Dispositivo
# =====================================================================
csv_maestros = """identificador_nodo,tipo_dispositivo,categoria,estado_operativo
NODO_ESP32_CENTRAL,Microcontrolador,Control,Activo
SENSOR_ACS712_PANEL,Sensor de Corriente,Medición,Activo
MODULO_RELEVADOR_LUCES,Actuador,Consumo,Inactivo
MOTOR_SERVOS_PANEL,Actuador,Control Posición,Activo"""
with open('3_datos_maestros_catalogo.csv', 'w', encoding='utf-8') as f:
    f.write(csv_maestros)


# =====================================================================
# 4. DATOS TRANSACCIONALES
# Operaciones en tiempo real (eventos de control y encendido/apagado).
# =====================================================================
datos_transaccionales = [
    {"timestamp": "2026-05-23T12:01:15Z", "usuario": "App_Movil", "accion": "Apagar Panel Solar", "resultado": "Exitoso"},
    {"timestamp": "2026-05-23T12:15:30Z", "usuario": "Sistema_ESP32", "accion": "Activar Ventilador (MOSFET)", "resultado": "Exitoso"}
]
with open('4_datos_transaccionales_eventos.json', 'w', encoding='utf-8') as f:
    json.dump(datos_transaccionales, f, indent=4, ensure_ascii=False)


# =====================================================================
# 5. FUENTES EXTERNAS: APIs Y SERVICIOS WEB
# Datos obtenidos para correlacionar el clima con la eficiencia.
# =====================================================================
api_clima_mock = {
  "fuente": "OpenWeather API",
  "ciudad": "León, Guanajuato",
  "prediccion_climatica": {
    "condicion": "Soleado",
    "temperatura_max_c": 32,
    "nubosidad_pct": 10,
    "recomendacion_sistema": "Óptimo para captación solar"
  }
}
with open('5_fuente_api_clima_externa.json', 'w', encoding='utf-8') as f:
    json.dump(api_clima_mock, f, indent=4, ensure_ascii=False)


# =====================================================================
# 6. LOGS DE SISTEMAS
# Registros para detectar fallas eléctricas o anomalías.
# =====================================================================
logs_sistema = """[INFO] 2026-05-23 11:50:00 - Sistema iniciado correctamente.
[INFO] 2026-05-23 11:55:00 - Conexión MQTT establecida con éxito.
[WARNING] 2026-05-23 12:02:10 - Anomalía detectada: Caída repentina de voltaje en Panel Solar.
[ERROR] 2026-05-23 12:05:00 - Falla eléctrica: Pérdida de comunicación con el sensor DHT11. Intentando reconectar..."""
with open('6_fuente_logs_anomalias.log', 'w', encoding='utf-8') as f:
    f.write(logs_sistema)

print("¡Se han generado los 6 archivos que representan tus Tipos y Fuentes de Datos!")
print("Revisa el ícono de la carpeta a la izquierda para descargarlos.")

¡Se han generado los 6 archivos que representan tus Tipos y Fuentes de Datos!
Revisa el ícono de la carpeta a la izquierda para descargarlos.
